In [ ]:
!pip install -q -U sentence-transformers transformers tokenizers huggingface-hub safetensors qdrant_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 38.4 MB/s eta 0:00:00


## 2. Import và kiểm tra phiên bản


In [ ]:

import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch
from packaging import version
from tqdm.auto import tqdm

import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence_transformers:', sentence_transformers.__version__)

# if version.parse(transformers.__version__).major >= 5:
#     raise RuntimeError(
#         'transformers >= 5 đang được dùng. GTE-Multilingual có lỗi position_ids với transformers>=5. '
#         'Hãy chạy lại cell cài đặt, restart runtime, rồi chạy lại notebook.'
#     )

DEVICE_CUDA_AVAILABLE = torch.cuda.is_available()
print('CUDA available:', DEVICE_CUDA_AVAILABLE)
if DEVICE_CUDA_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Đã login Hugging Face.")
else:
    print("Chưa có HF_TOKEN.")



torch: 2.11.0+cu128
transformers: 5.13.1
sentence_transformers: 5.6.0
CUDA available: True
GPU: Tesla T4
Đã login Hugging Face.


## 3. Cấu hình chung


In [ ]:

DATA_FILE = Path('/content/full_DVSK_data_aiteamvn_256.json')
MAX_SEQ_LENGTH = 512


# Sau khi ổn, đổi thành ['bge_m3', 'aiteamvn', 'gte_multilingual'].
RUN_MODELS = ['aiteamvn_v2']

# Đặt True nếu muốn xóa và tạo lại collection đang chạy.
RECREATE_COLLECTION = True

UPSERT_BATCH_SIZE = 64

MODEL_CONFIGS = {
    # 'bge_m3': {
    #     'model_id': 'BAAI/bge-m3',
    #     'collection_name': 'dvsktt_bge_m3_chunks_256',
    #     'trust_remote_code': False,
    #     'device': 'cuda' if DEVICE_CUDA_AVAILABLE else 'cpu',
    #     'batch_size': 16 if DEVICE_CUDA_AVAILABLE else 4,
    # },
    # 'aiteamvn': {
    #     'model_id': 'AITeamVN/Vietnamese_Embedding',
    #     'collection_name': 'dvsktt_aiteamvn_chunks_256',
    #     'trust_remote_code': False,
    #     'device': 'cuda' if DEVICE_CUDA_AVAILABLE else 'cpu',
    #     'batch_size': 16 if DEVICE_CUDA_AVAILABLE else 4,
    # },
    # 'gte_multilingual': {
    #     'model_id': 'Alibaba-NLP/gte-multilingual-base',
    #     'collection_name': 'dvsktt_gte_multilingual_chunks_256',
    #     'trust_remote_code': True,
    #     # Giữ CPU cho GTE để tránh lỗi CUDA/CUBLAS trên Colab.
    #     'device': 'cpu',
    #     'batch_size': 2,
    # },
    'aiteamvn_v2': {
        'model_id': 'AITeamVN/Vietnamese_Embedding_v2',
        'collection_name': 'dvsktt_aiteamvn_v2_chunks_256',
        'trust_remote_code': False,
        'device': 'cuda' if DEVICE_CUDA_AVAILABLE else 'cpu',
        'batch_size': 16 if DEVICE_CUDA_AVAILABLE else 4,
    }
}

for key in RUN_MODELS:
    if key not in MODEL_CONFIGS:
        raise ValueError(f'Model key không hợp lệ: {key}')

# print('DATA_FILE:', DATA_FILE)
# print('MAX_SEQ_LENGTH:', MAX_SEQ_LENGTH)
# print('RUN_MODELS:', RUN_MODELS)


## 4. Kết nối Qdrant

Notebook ưu tiên đọc `QDRANT_URL` và `QDRANT_API_KEY` từ Colab Secrets. Nếu không có Colab Secrets, nó sẽ đọc từ biến môi trường.


In [ ]:

def get_secret_or_env(name: str) -> Optional[str]:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)

QDRANT_URL = get_secret_or_env('QDRANT_URL')
QDRANT_API_KEY = get_secret_or_env('QDRANT_API_KEY')

if not QDRANT_URL:
    raise RuntimeError('Thiếu QDRANT_URL. Hãy thêm vào Colab Secrets hoặc biến môi trường.')
if not QDRANT_API_KEY:
    raise RuntimeError('Thiếu QDRANT_API_KEY. Hãy thêm vào Colab Secrets hoặc biến môi trường.')

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)
print('Đã kết nối Qdrant.')


Đã kết nối Qdrant.


## 5. Đọc và kiểm tra dữ liệu JSON

Notebook yêu cầu mỗi record có `chunk_id` và `embed_input`. Các trường payload được lấy đúng theo schema hiện tại của bạn.


In [ ]:

def load_records(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy file: {path}')
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError('File JSON phải là list các chunk.')
    return data

records = load_records(DATA_FILE)
print('Tổng records:', len(records))
print('Ví dụ keys:', sorted(records[0].keys()))

bad = []
for i, r in enumerate(records):
    if not r.get('chunk_id'):
        bad.append((i, 'missing chunk_id'))
    if not isinstance(r.get('embed_input'), str) or not r.get('embed_input').strip():
        bad.append((i, 'missing/empty embed_input'))

if bad:
    print('Một số lỗi đầu tiên:', bad[:10])
    raise ValueError(f'Dữ liệu có {len(bad)} record lỗi. Cần sửa trước khi embedding.')

ids = [r['chunk_id'] for r in records]
if len(ids) != len(set(ids)):
    raise ValueError('Có chunk_id bị trùng. Qdrant sẽ ghi đè point nếu ID trùng.')

lengths = [len(r['embed_input']) for r in records]
print('Độ dài ký tự embed_input min/mean/max:', min(lengths), round(float(np.mean(lengths)), 2), max(lengths))
print('Record đầu tiên chunk_id:', records[0]['chunk_id'])


NameError: name 'DATA_FILE' is not defined

## 6. Hàm tạo payload Qdrant

Chỉ lưu các trường cần thiết để retrieval trả về evidence. Không lưu `embed_input`, `segmented_text`, `overlap_text`, `section_title`.


In [ ]:

def build_payload(record: Dict[str, Any]) -> Dict[str, Any]:
    return {
        'book_name': record.get('book_name'),
        'pages': record.get('pages', []),
        'raw_text': record.get('raw_text'),
        'headers': record.get('headers', {}),
        'footnotes': record.get('footnotes', {}),
        'token_count': record.get('token_count'),
    }


## 7. Vá lỗi position_ids cho GTE-Multilingual

Theo HuggingFace Discussion của model, `transformers>=5.0` có thể làm hỏng non-persistent buffer `position_ids`. Notebook đã pin `transformers<5`, nhưng vẫn vá lại buffer sau khi load GTE để tránh cache hoặc môi trường Colab gây lỗi.


In [ ]:

def fix_gte_position_ids_if_possible(model: SentenceTransformer) -> None:
    try:
        first_module = model._first_module()
        auto_model = getattr(first_module, 'auto_model', None)
        if auto_model is None:
            print('[GTE] Không tìm thấy auto_model trong SentenceTransformer; bỏ qua vá position_ids.')
            return

        embeddings = getattr(auto_model, 'embeddings', None)
        if embeddings is None or not hasattr(embeddings, 'register_buffer'):
            print('[GTE] Không tìm thấy embeddings có register_buffer; bỏ qua vá position_ids.')
            return

        max_pos = int(getattr(auto_model.config, 'max_position_embeddings', 8192))
        device = next(auto_model.parameters()).device
        position_ids = torch.arange(max_pos, dtype=torch.long, device=device)
        embeddings.register_buffer('position_ids', position_ids, persistent=True)

        check = embeddings.position_ids
        print('[GTE] Đã vá position_ids:', tuple(check.shape), 'min=', int(check.min()), 'max=', int(check.max()))
    except Exception as e:
        print('[GTE] CẢNH BÁO: không vá được position_ids:', repr(e))


## 8. Load model embedding

Cả 3 model đều dùng model gốc từ HuggingFace. GTE dùng `SentenceTransformer(..., trust_remote_code=True)` đúng theo model card.


In [ ]:

def load_embedding_model(model_key: str) -> SentenceTransformer:
    cfg = MODEL_CONFIGS[model_key]
    print('=' * 80)
    print('Bắt đầu model:', model_key)
    print('Model ID:', cfg['model_id'])
    print('Collection:', cfg['collection_name'])
    print('Device:', cfg['device'])

    model = SentenceTransformer(
        cfg['model_id'],
        trust_remote_code=cfg['trust_remote_code'],
        device=cfg['device'],
    )
    model.max_seq_length = MAX_SEQ_LENGTH

    if model_key == 'gte_multilingual':
        fix_gte_position_ids_if_possible(model)

    dim = int(model.get_sentence_embedding_dimension())
    print('Embedding dimension:', dim)
    print('Max sequence length:', model.max_seq_length)
    return model


## 9. Kiểm tra model trước khi upsert

Nếu model sinh vector lỗi, notebook dừng ngay và không upsert vector rác vào Qdrant.


In [ ]:

def sanitize_vectors(vectors: Any, expected_dim: int, model_key: str) -> np.ndarray:
    arr = np.asarray(vectors, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    if arr.ndim != 2:
        raise ValueError(f'[{model_key}] Vector phải có 2 chiều, nhận shape={arr.shape}')
    if arr.shape[1] != expected_dim:
        raise ValueError(f'[{model_key}] Sai dimension: nhận {arr.shape[1]}, expected {expected_dim}')
    if not np.isfinite(arr).all():
        raise ValueError(f'[{model_key}] Vector có NaN/Inf. Dừng để tránh upsert vector lỗi.')
    norms = np.linalg.norm(arr, axis=1)
    if np.any(norms < 1e-8):
        raise ValueError(f'[{model_key}] Có vector gần bằng 0. Dừng để tránh retrieval lỗi.')
    return np.ascontiguousarray(arr, dtype=np.float32)


def sanity_check_model(model_key: str, model: SentenceTransformer) -> int:
    dim = int(model.get_sentence_embedding_dimension())
    sample_texts = [
        'Vua Nghiêu sai Hy thị đến ở Nam Giao để định đất Giao Chỉ ở phương Nam.',
        'Đại Việt Sử Ký Toàn Thư ghi chép nhiều sự kiện lịch sử Việt Nam.',
        records[0]['embed_input'],
    ]
    vectors = model.encode(
        sample_texts,
        batch_size=1 if model_key == 'gte_multilingual' else 4,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    arr = sanitize_vectors(vectors, dim, model_key)
    print(f'[{model_key}] Sanity check OK. shape={arr.shape}, norm={np.linalg.norm(arr[0]):.4f}')
    return dim


## 10. Tạo hoặc kiểm tra collection Qdrant


In [ ]:

def collection_exists(name: str) -> bool:
    try:
        qdrant.get_collection(name)
        return True
    except Exception:
        return False


def get_existing_vector_size(collection_name: str) -> Optional[int]:
    try:
        info = qdrant.get_collection(collection_name)
        vectors_cfg = info.config.params.vectors
        if hasattr(vectors_cfg, 'size'):
            return int(vectors_cfg.size)
        if isinstance(vectors_cfg, dict):
            first = next(iter(vectors_cfg.values()))
            return int(first.size)
    except Exception:
        return None
    return None


def prepare_collection(collection_name: str, vector_size: int) -> None:
    exists = collection_exists(collection_name)
    if exists and RECREATE_COLLECTION:
        print('Xóa collection cũ:', collection_name)
        qdrant.delete_collection(collection_name)
        exists = False

    if exists:
        old_size = get_existing_vector_size(collection_name)
        if old_size is not None and old_size != vector_size:
            raise ValueError(
                f'Collection {collection_name} đang có vector_size={old_size}, nhưng model hiện tại là {vector_size}. '
                f'Hãy đặt RECREATE_COLLECTION=True để tạo lại.'
            )
        print('Collection đã tồn tại, sẽ upsert tiếp:', collection_name)
        return

    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )
    print('Đã tạo collection:', collection_name)


## 11. Encode và upsert vào Qdrant


In [ ]:

def encode_texts(model_key: str, model: SentenceTransformer, texts: List[str]) -> np.ndarray:
    cfg = MODEL_CONFIGS[model_key]
    dim = int(model.get_sentence_embedding_dimension())
    vectors = model.encode(
        texts,
        batch_size=cfg['batch_size'],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return sanitize_vectors(vectors, dim, model_key)


def make_points(batch_records: List[Dict[str, Any]], vectors: np.ndarray) -> List[PointStruct]:
    points = []
    for record, vector in zip(batch_records, vectors):
        points.append(
            PointStruct(
                id=record['chunk_id'],
                vector=[float(x) for x in vector.tolist()],
                payload=build_payload(record),
            )
        )
    return points


def upsert_records_for_model(model_key: str) -> None:
    cfg = MODEL_CONFIGS[model_key]
    model = load_embedding_model(model_key)
    vector_size = sanity_check_model(model_key, model)
    prepare_collection(cfg['collection_name'], vector_size)

    total_batches = (len(records) + UPSERT_BATCH_SIZE - 1) // UPSERT_BATCH_SIZE
    print('Encode batch_size:', cfg['batch_size'])
    print('Upsert batch_size:', UPSERT_BATCH_SIZE)

    for start in tqdm(range(0, len(records), UPSERT_BATCH_SIZE), total=total_batches, desc=f'Upserting {model_key}'):
        batch_records = records[start:start + UPSERT_BATCH_SIZE]
        texts = [r['embed_input'] for r in batch_records]
        vectors = encode_texts(model_key, model, texts)
        points = make_points(batch_records, vectors)
        qdrant.upsert(collection_name=cfg['collection_name'], points=points, wait=True)

    print(f'Hoàn tất model {model_key}: {cfg["collection_name"]}')

    # Giải phóng RAM/VRAM sau mỗi model.
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 12. Chạy embedding

Khuyến nghị: chạy `RUN_MODELS=['gte_multilingual']` trước. Khi GTE đã ổn, chạy lại với cả 3 model.


In [ ]:

for model_key in RUN_MODELS:
    upsert_records_for_model(model_key)

print('Đã hoàn tất toàn bộ RUN_MODELS.')


Bắt đầu model: aiteamvn_v2
Model ID: AITeamVN/Vietnamese_Embedding_v2
Collection: dvsktt_aiteamvn_v2_chunks_256
Device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

/tmp/ipykernel_2285/302319470.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = int(model.get_sentence_embedding_dimension())
/tmp/ipykernel_2285/752352664.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = int(model.get_sentence_embedding_dimension())


Embedding dimension: 1024
Max sequence length: 512
[aiteamvn_v2] Sanity check OK. shape=(3, 1024), norm=1.0000
Đã tạo collection: dvsktt_aiteamvn_v2_chunks_256
Encode batch_size: 4
Upsert batch_size: 64


Upserting aiteamvn_v2:   0%|          | 0/84 [00:00<?, ?it/s]

/tmp/ipykernel_2285/453029141.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = int(model.get_sentence_embedding_dimension())


Hoàn tất model aiteamvn_v2: dvsktt_aiteamvn_v2_chunks_256
Đã hoàn tất toàn bộ RUN_MODELS.


## 13. Kiểm tra số lượng point trong collection


In [ ]:

for model_key in RUN_MODELS:
    collection_name = MODEL_CONFIGS[model_key]['collection_name']
    info = qdrant.get_collection(collection_name)
    print(collection_name, 'points_count =', info.points_count)


dvsktt_aiteamvn_v2_chunks_256 points_count = 5343
